In [1]:
# Install libraries
!pip install pandas
%pip install nbconvert


Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import pandas as pd
from datetime import datetime
from decimal import Decimal, ROUND_HALF_UP

In [3]:
# Get the current directory of the notebook
notebook_dir = os.getcwd()

# Get the file path of data
tutor_assignments_path = os.path.join(notebook_dir, "data", "raw", "tutor_assignments_raw.xlsx")
print(tutor_assignments_path)

/Users/huien/Documents/opus-tuition/src/data_cleaning/data/raw/tutor_assignments_raw.xlsx


#### Read the dataset

In [4]:
# Read the dataset
df = pd.read_excel(tutor_assignments_path, header = 7)

# Get first five row of the dataset
df.head()

,Assignment ID,Tutor Name,Student Name,Subject,Level,Hourly Rate (SGD),Start Date,Status,Contact Email
0,TAS-001,Ahmad Rizwan,Lim Wei Jie,Mathematics,Secondary 3,55.0,01/05/2025,Active,ahmad.r@tutors.com
1,TAS-002,Priya Nair,Tan Xiu Ying,english,Primary 5,45.0,2025-03-12,active,priya.n@tutors.com
2,TAS-003,Jason Teo,Muhammad Hafiz,MATHEMATICS,Secondary 4,60.0,12 Feb 2025,ACTIVE,jason.t@tutors.com
3,TAS-004,Sarah Lim,Chloe Wong,Science,Primary 6,50.0,03/18/2025,Active,NaN
4,TAS-005,Ahmad Rizwan,Lim Wei Jie,Mathematics,Secondary 3,55.0,01/05/2025,Active,ahmad.r@tutors.com


#### Analyse the dataset

In [5]:
# Get the number of columns and rows of the dataset
df.shape

(15, 9)

In [6]:
# Get the column names
column_names = list(df.columns)
print(column_names)

['Assignment ID', 'Tutor Name', 'Student Name', 'Subject', 'Level', 'Hourly Rate (SGD)', 'Start Date', 'Status', 'Contact Email']


In [7]:
# Check for data types
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Assignment ID      13 non-null     str    
 1   Tutor Name         13 non-null     str    
 2   Student Name       12 non-null     str    
 3   Subject            13 non-null     str    
 4   Level              13 non-null     str    
 5   Hourly Rate (SGD)  13 non-null     float64
 6   Start Date         13 non-null     str    
 7   Status             13 non-null     str    
 8   Contact Email      12 non-null     str    
dtypes: float64(1), str(8)
memory usage: 1.2 KB


In [8]:
# Get distinct value for subject
subject = list(df["Subject"].unique())
print(subject)

['Mathematics', 'english', 'MATHEMATICS', 'Science', nan, 'Chemistry', 'Physics', 'maths', 'English', 'Biology']


In [9]:
# Get distinct value for level
level = list(df["Level"].unique())
print(level)

['Secondary 3', 'Primary 5', 'Secondary 4', 'Primary 6', nan, 'Junior College 1', 'Secondary 2', 'Secondary 1', 'Primary 4']


In [10]:
# Get distinct value for payment status
status = list(df["Status"].unique())
print(status)

['Active', 'active', 'ACTIVE', nan, 'Inactive', 'Pending']


In [11]:
# Identify duplicated rows
duplicate_rows = df[df.duplicated(subset=["Assignment ID"], keep=False)]
print(duplicate_rows)

   Assignment ID Tutor Name Student Name Subject Level  Hourly Rate (SGD)  \
5            NaN        NaN          NaN     NaN   NaN                NaN   
10           NaN        NaN          NaN     NaN   NaN                NaN   

   Start Date Status Contact Email  
5         NaN    NaN           NaN  
10        NaN    NaN           NaN  


In [12]:
# Identify duplicated rows
duplicate_rows = df[df.duplicated(subset=["Tutor Name", "Student Name", "Start Date"], keep=False)]
print(duplicate_rows)

   Assignment ID    Tutor Name  Student Name      Subject        Level  \
0        TAS-001  Ahmad Rizwan   Lim Wei Jie  Mathematics  Secondary 3   
1        TAS-002    Priya Nair  Tan Xiu Ying      english    Primary 5   
4        TAS-005  Ahmad Rizwan   Lim Wei Jie  Mathematics  Secondary 3   
5            NaN           NaN           NaN          NaN          NaN   
9        TAS-009    Priya Nair  Tan Xiu Ying      English    Primary 5   
10           NaN           NaN           NaN          NaN          NaN   
14       TAS-013  Ahmad Rizwan   Lim Wei Jie  Mathematics  Secondary 3   

    Hourly Rate (SGD)  Start Date  Status       Contact Email  
0                55.0  01/05/2025  Active  ahmad.r@tutors.com  
1                45.0  2025-03-12  active  priya.n@tutors.com  
4                55.0  01/05/2025  Active  ahmad.r@tutors.com  
5                 NaN         NaN     NaN                 NaN  
9                45.0  2025-03-12  Active  priya.n@tutors.com  
10                NaN  

#### Clean dataset

In [13]:
# Remove duplicates, leaving only last occurrence
df_cleaned = df.drop_duplicates(subset=["Tutor Name", "Student Name", "Start Date"], keep="last")

In [14]:
# Drop blank rows with all values are nulls
df_cleaned = df_cleaned.dropna(how="all")

In [15]:
import pandas as pd
from datetime import datetime

def parse_single_column(df: pd.DataFrame, col_name: str) -> pd.DataFrame:
    # 1. Define the exact formats found in your Opus Tuition export image
    expected_formats = [
        "%Y-%m-%d",   # 2025-03-12
        "%m/%d/%Y",   # 01/05/2025
        "%d-%m-%Y",   # 25-04-2025
        "%d/%m/%y",   # 25/04/26
        "%d %b %Y",   # 12 Feb 2025
        "%B %d, %Y",  # July 7, 2025
        "%d-%b-%Y"    # 15-Oct-2025
    ]

    parsed_values = []
    error_log = []

    # 2. Loop only through the target column
    for idx, raw_val in df[col_name].items():
        if pd.isna(raw_val) or str(raw_val).strip() in ["", "None", "NaN"]:
            parsed_values.append(pd.NaT)
            continue

        clean_val = str(raw_val).strip()
        success = False

        for fmt in expected_formats:
            try:
                parsed_values.append(datetime.strptime(clean_val, fmt))
                success = True
                break
            except ValueError:
                continue

        if not success:
            # Capture specific row index and offending value
            error_log.append(f"Row {idx}: '{raw_val}' cannot be parsed.")
            parsed_values.append(pd.NaT)

    # 3. If any row failed, stop everything and throw a specific error
    if error_log:
        raise ValueError(
            f"Data Integrity Error in column [{col_name}].\n"
            f"Details:\n" + "\n".join(error_log)
        )

    # 4. If all passed, overwrite only that single column with the datetime type
    df[col_name] = parsed_values
    return df

df_cleaned = parse_single_column(df_cleaned, "Start Date")

In [16]:
def handle_string_column(series):
    # Ensure we are treating the data as string
    s = series.astype("string")
    # Strip leading, trailing & replace internal excess whitespaces
    s = s.str.strip().str.replace(r"\s+", " ", regex=True)
    # Lowercase and capitalize
    s = s.str.lower().str.capitalize()
    return s

# Define the list of columns for cleaning
target_cols = ["Subject", "Status"]

# Apply the function to all target columns at once
df_cleaned[target_cols] = df_cleaned[target_cols].apply(handle_string_column)

In [17]:
# Replace specific abbreviations
df_cleaned["Subject"] = df_cleaned["Subject"].replace({"Maths": "Mathematics"})

In [18]:
# Extract numbers, periods, and commas. Handle missing values safely.
df_cleaned["Hourly Rate (SGD)"] = (
    df_cleaned["Hourly Rate (SGD)"]
    .astype(str)
    .str.extract(r"([\d.,]+)")[0]
    .str.replace(",", "", regex=False)
)

# Convert to Decimal and strictly round to 2 decimal places
def to_rounded_decimal(val):
    # Catch null and empty strings
    if pd.isna(val) or val in ["nan", ""]:
        return None
    try:
        # Quantize ensures exactly two decimal places (.01)
        return Decimal(val).quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)
    except Exception:
        return None

df_cleaned["Hourly Rate (SGD)"] = df_cleaned["Hourly Rate (SGD)"].apply(to_rounded_decimal)

In [19]:
# Convert selected columns to the modern string dtype
df_cleaned = df_cleaned.astype({
    "Assignment ID": "string",
    "Tutor Name": "string",
    "Student Name": "string",
    "Level": "string",
    "Contact Email": "string"
})

#### Validate dataset

In [20]:
# Check data frame's summary
df_cleaned.info()

<class 'pandas.DataFrame'>
Index: 10 entries, 2 to 14
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Assignment ID      10 non-null     string        
 1   Tutor Name         10 non-null     string        
 2   Student Name       9 non-null      string        
 3   Subject            10 non-null     string        
 4   Level              10 non-null     string        
 5   Hourly Rate (SGD)  10 non-null     object        
 6   Start Date         10 non-null     datetime64[us]
 7   Status             10 non-null     string        
 8   Contact Email      9 non-null      string        
dtypes: datetime64[us](1), object(1), string(7)
memory usage: 800.0+ bytes


In [21]:
# Get distinct value for subject
subject = list(df_cleaned["Subject"].unique())
print(subject)

['Mathematics', 'Science', 'Chemistry', 'Physics', 'English', 'Biology']


In [22]:
# Print the final cleaned results
df_cleaned

,Assignment ID,Tutor Name,Student Name,Subject,Level,Hourly Rate (SGD),Start Date,Status,Contact Email
2,TAS-003,Jason Teo,Muhammad Hafiz,Mathematics,Secondary 4,60.00,2025-02-12,Active,jason.t@tutors.com
3,TAS-004,Sarah Lim,Chloe Wong,Science,Primary 6,50.00,2025-03-18,Active,<NA>
6,TAS-006,David Kwok,Ethan Ng,Chemistry,Junior College 1,70.00,2025-04-25,Inactive,david.k@tutors.com
7,TAS-007,Nurul Huda,<NA>,Physics,Secondary 2,50.00,2025-06-01,Active,nurul.h@tutors.com
8,TAS-008,Marcus Chan,Aisha Binte Yusof,Mathematics,Secondary 1,45.00,2025-07-07,Active,marcus.c@tutors.com
9,TAS-009,Priya Nair,Tan Xiu Ying,English,Primary 5,45.00,2025-03-12,Active,priya.n@tutors.com
11,TAS-010,Li Mei,Darren Foo,Biology,Secondary 3,55.00,2025-08-20,Active,li.mei@tutors.com
12,TAS-011,Jason Teo,Raeanne Ong,Mathematics,Secondary 4,60.00,2025-09-01,Pending,jason.t@tutors.com
13,TAS-012,Sarah Lim,Jordan Tan,Science,Primary 4,45.00,2025-10-15,Active,sarah.l@tutors.com
14,TAS-013,Ahmad Rizwan,Lim Wei Jie,Mathematics,Secondary 3,55.00,2025-01-05,Active,ahmad.r@tutors.com
